In [ ]:
import numpy as np
import torch
if torch.cuda.is_available():
    device = torch.device('cuda:0')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

### Graph MNIST

In [ ]:
from torch_geometric import datasets
from torch_geometric.transforms import AddRandomWalkPE, AddLaplacianEigenvectorPE
dataset_name = "MNIST"
PE_DIM = 20

dataset = datasets.GNNBenchmarkDataset(
    root="/path/to/data",
    name=dataset_name,
)
print(dataset.data, dataset.num_node_features, dataset.num_edge_features, dataset.num_classes)

pe_transform = AddRandomWalkPE(walk_length=PE_DIM, attr_name='pe')
dataset = datasets.GNNBenchmarkDataset(
    root="/path/to/data_rwpe",
    name=dataset_name,
    pre_transform=pe_transform,
    # force_reload=True,
)
print(dataset.data, dataset.num_node_features, dataset.num_edge_features, dataset.num_classes)

pe_transform = AddLaplacianEigenvectorPE(k=PE_DIM, attr_name='pe', is_undirected=True)
dataset = datasets.GNNBenchmarkDataset(
    root="/path/to/data_lappe",
    name=dataset_name,
    pre_transform=pe_transform,
    # force_reload=True,
)
print(dataset.data, dataset.num_node_features, dataset.num_edge_features, dataset.num_classes)

### Superpixel Image MNIST

In [2]:
from torchvision import datasets, transforms
from skimage.segmentation import slic

class SuperpixelTransform:
    """
    A custom transform to convert a 28*28 MNIST grayscale image into its superpixel-averaged representation.
    """
    def __init__(self, n_segments=75):
        self.n_segments = n_segments

    def __call__(self, img):
        img_np = np.array(img, dtype=np.float32) / 255.0
        segments = slic(img_np, n_segments=self.n_segments, slic_zero=True, channel_axis=None)
        new_img = np.zeros_like(img_np)
        for seg_val in np.unique(segments):
            mask = (segments == seg_val)
            mean_intensity = img_np[mask].mean(axis=0)
            new_img[mask] = mean_intensity
        return new_img

transform = transforms.Compose([
    SuperpixelTransform(n_segments=75),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = datasets.MNIST(root='/path/to/data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='/path/to/data', train=False, transform=transform, download=True)

In [ ]:
def precompute_dataset(dataset):
    data_list = []
    labels_list = []
    for idx in range(len(dataset)):
        img, label = dataset[idx]
        data_list.append(img)
        labels_list.append(label)
        if idx % 10000 == 0:
            print(f"Processed {idx} samples...")
    data_tensor = torch.stack(data_list)
    labels_tensor = torch.tensor(labels_list)
    return data_tensor, labels_tensor
train_data, train_labels = precompute_dataset(train_dataset)
test_data, test_labels = precompute_dataset(test_dataset)
torch.save((train_data, train_labels), "/path/to/data/mnist_train_superpixel.pt")
torch.save((test_data, test_labels), "/path/to/data/mnist_test_superpixel.pt")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_data, train_labels = torch.load("/path/to/data/mnist_train_superpixel.pt", weights_only=False)
test_data, test_labels = torch.load("/path/to/data/mnist_test_superpixel.pt", weights_only=False)
train_dataset = TensorDataset(train_data, train_labels)
test_dataset = TensorDataset(test_data, test_labels)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)